# French-to-English fine-tuning workflow

Use the project's WSL Python kernel. This notebook calls CLI stages in separate processes and reads their saved artifacts. Each process releases its model memory when the stage ends.

The normal profile trains on 20,000 French-English pairs split equally between Books and OPUS-100. It compares the original Marian model with the validation-best fine-tuned checkpoint. Start with the smoke profile to exercise the workflow. See `docs/workflow.md` and `docs/evaluation.md` for setup and metric definitions.

Notebook output can contain local paths, environment details, hardware information, and benchmark timings. Clear saved outputs before sharing this notebook. See the sharing guidance in `docs/workflow.md` for preparing a public results summary.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

assert platform.system() == 'Linux' and 'microsoft' in platform.release().lower(), 'Select the WSL kernel'
PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
assert (PROJECT / 'pyproject.toml').exists(), 'Open this notebook from the project folder'
PROFILE = PROJECT / 'configs' / 'smoke.yaml'  # Use local.yaml for the normal experiment.
RUN = None  # Set to the run directory you want to inspect or resume.
EXECUTE_STAGES = False  # Enable to run pipeline stages.
env = dict(os.environ, PYTHONPATH=str(PROJECT / 'src'), TORCH_DISABLE_NATIVE_JIT='1')

def cli(*args, capture=False):
    return subprocess.run([sys.executable, '-m', 'fr_en_ft', *map(str, args)], cwd=PROJECT, env=env,
                          check=True, text=True, capture_output=capture)

def artifact(name):
    assert RUN is not None, 'Select or create a run first'
    return json.loads((RUN / name).read_text(encoding='utf-8'))

def stage(name):
    if EXECUTE_STAGES:
        extra = ['--resume-training'] if name == 'train' and (RUN / 'checkpoints').exists() else []
        cli(name, '--run-dir', RUN, *extra)
    else:
        print(f'Stage execution disabled: {name}')

print(sys.executable)

## Environment and run identity
Check the interpreter, CUDA, and dependency versions before creating a run. Existing artifacts can be inspected with stage execution disabled.

In [ ]:
display(json.loads(cli('doctor', capture=True).stdout))
if EXECUTE_STAGES and RUN is None:
    RUN = Path(cli('init', '--config', PROFILE, capture=True).stdout.strip())
print('Run:', RUN)

## Download and prepare
Downloads use pinned Hugging Face revisions and stay outside the active-time budget. Preparation normalizes text, removes bad/duplicate pairs, prevents French-source overlap across splits, and excludes inputs or references beyond the configured token limits.

Books uses source-hash splits, not book-level splits. OPUS-100 preserves official holdouts and has no reliable spoken-language labels. The manifest distinguishes full-corpus exclusions from tokenized-candidate exclusions.

In [ ]:
stage('download')
stage('prepare')
if RUN is not None and (RUN / 'data_manifest.json').exists():
    manifest = artifact('data_manifest.json')
    display(pd.DataFrame([{'split': name, 'examples': len(rows)} for name, rows in manifest['splits'].items()]))
    display(pd.Series(manifest['stats'], name='count').to_frame())
    print('Data fingerprint:', manifest['data_fingerprint'])

## Calibration and baseline
Calibration checks resource use and estimates the work that fits the configured budget. Its training updates are discarded. Baseline evaluation reloads the original weights and scores the fixed test set. Test quality never chooses training settings. The calibration output below is for local use.

In [ ]:
stage('benchmark')
if RUN is not None and (RUN / 'benchmark.json').exists():
    display(artifact('benchmark.json'))
stage('baseline')
if RUN is not None and (RUN / 'baseline_metrics.json').exists():
    display(artifact('baseline_metrics.json')['overall'])

## Train and checkpoint
The pipeline freezes a measured step budget before training. The default is full fine-tuning with AdamW, learning rate 2e-5, dynamic padding, and mixed precision. The best validation chrF++ chooses the final checkpoint.

Recovery checkpoints include optimizer, scheduler, random states, scaler when applicable, and training control. Complete bundles are published atomically and checksummed. Resume with `python -m fr_en_ft run --resume PATH`, replacing `PATH` with your run directory. Other applications can affect the runtime estimate.

In [ ]:
stage('train')
if RUN is not None and (RUN / 'training_plan.json').exists():
    display(artifact('training_plan.json'))
if RUN is not None and (RUN / 'events.jsonl').exists():
    events = pd.read_json(RUN / 'events.jsonl', lines=True)
    display(events[events['event'] == 'trainer'].tail(20))

## Final evaluation
Compare generated translations using SacreBLEU, chrF++, TER, BERTScore, token-weighted NLL, and perplexity. Quality results include source and length breakdowns, diagnostic flags, and paired bootstrap intervals. Review the local report before sharing it, since it also contains runtime metadata.

Intervals describe test-sample uncertainty, not training-seed variation. OPUS material may overlap the base model's pretraining. The two examples below were selected before training and are only for observation.

In [ ]:
stage('final')
stage('report')
if RUN is not None and (RUN / 'comparison.csv').exists():
    display(pd.read_csv(RUN / 'comparison.csv'))
    display(artifact('confidence_intervals.json'))
    display(pd.DataFrame(artifact('examples.json')))
    display(Image(filename=str(RUN / 'learning-curves.png')))
    display(Markdown((RUN / 'report.md').read_text(encoding='utf-8')))